In [ ]:
from ultralytics import YOLO
import cv2

# bigger model helps small objects; try 'yolo11l.pt' or 'yolo11x.pt'
model = YOLO("yolo11n-obb.pt")

video = "../Dataset/Galatsi_Data_Semester_Project_archive/DJI_0808.MOV"
img_path = '../first_frame_moving.png'
img = cv2.imread(img_path)

results = model.track(
    source=img,
    imgsz=2560,             # up from 640; improves small-object recall
    #imgsz=3840,
    # rect=True,
    #conf=0.50,              # lower to catch tiny cars
    iou=0.7,
    classes=[9,10],        # small vehicules = 10, big vehicules = 9
    stream=False,           # set True to iterate frames
    show=False,               # aef
    save=False,              # save=True to save
    show_labels=False,
)


0: 1440x2560 17 large vehicles, 261 small vehicles, 265.8ms
Speed: 22.8ms preprocess, 265.8ms inference, 94.6ms postprocess per image at shape (1, 3, 1440, 2560)


In [17]:
import cv2
from ultralytics import YOLO
from patched_yolo_infer import auto_calculate_crop_values

# Load the image
img_path = "../first_frame_static.png"
img = cv2.imread(img_path)

#print(type(img))   # doit afficher <class 'numpy.ndarray'>

# Calculate the optimal crop size and overlap for an image
shape_x, shape_y, overlap_x, overlap_y = auto_calculate_crop_values(
    image=img, mode="network_based", model=YOLO("yolo11n.pt")
)

print(shape_x, shape_y, overlap_x, overlap_y)

960 540 25 25


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Num GPUs:", torch.cuda.device_count())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("MPS available:", torch.backends.mps.is_available())


CUDA available: True
Num GPUs: 1
Device: NVIDIA GeForce RTX 4070 Ti SUPER
MPS available: False


In [1]:

import supervision as sv, inspect
from supervision.tracker.byte_tracker.core import ByteTrack

print("supervision version:", sv.__version__)
print("ByteTrack module:", ByteTrack.__module__)
print("ByteTrack qualname:", ByteTrack.__qualname__)
print("Init signature:", inspect.signature(ByteTrack.__init__))
# Optional: full path to the module file
import importlib, os
m = importlib.import_module(ByteTrack.__module__)
print("Loaded from:", getattr(m, "__file__", "<no __file__>"))


supervision version: 0.26.1
ByteTrack module: supervision.tracker.byte_tracker.core
ByteTrack qualname: ByteTrack
Init signature: (self, track_activation_threshold: float = 0.25, lost_track_buffer: int = 30, minimum_matching_threshold: float = 0.8, frame_rate: int = 30, minimum_consecutive_frames: int = 1)
Loaded from: c:\Users\makss\Git\Galatsi-Semester-Project\.venv\Lib\site-packages\supervision\tracker\byte_tracker\core.py


In [1]:
import time
import sys
sys.path.append("C:/Users/makss/Git/Galatsi-Semester-Project/Code/test_patch_image_video.py")
import test_patch_image_video as tpiv

def benchmark_single_frame(frame, model, tiles=(4,3), imgsz=2560, obb=True, device="0"):
    times = {}

    # T0 - start
    t0 = time.time()
    tiles_rc = tpiv.make_tiles(frame.shape[0], frame.shape[1], *tiles, overlap_ratio=0.25)
    times["make_tiles"] = time.time() - t0

    all_dets = []
    t1 = time.time()
    for (x0, y0, x1, y1) in tiles_rc:
        tile = frame[y0:y1, x0:x1]
        results = model.predict(
            source=tile,
            imgsz=imgsz,
            conf=0.25,
            device=device,
            verbose=False,
            classes=[9, 10]
        )
        for res in results:
            dets = tpiv.extract_detections(res, x_offset=x0, y_offset=y0, task="obb" if obb else "detect")
            all_dets.extend(dets)
    times["yolo+extract"] = time.time() - t1

    t2 = time.time()
    merged = tpiv.nms_merge_fast(all_dets, iou_thr=0.5)
    times["nms_merge"] = time.time() - t2

    return merged, times


In [2]:
from ultralytics import YOLO
import cv2


model = YOLO("yolo11m-obb.pt")
cap = cv2.VideoCapture("../Dataset/Galatsi_Data_Semester_Project_archive/DJI_0004.MP4")
ret, frame = cap.read()

# maintenant tu peux appeler directement
merged, times = benchmark_single_frame(frame, model, tiles=(4,3), imgsz=2560, obb=True)
print(times)


{'make_tiles': 0.0, 'yolo+extract': 1.681147813796997, 'nms_merge': 0.030052661895751953}


In [4]:
import cv2, time
from ultralytics import YOLO
import sys
sys.path.append("C:/Users/makss/Git/Galatsi-Semester-Project/Code")

import test_patch_image_video as tpiv

video = "../Dataset/Galatsi_Data_Semester_Project_archive/DJI_0004.MP4"
ret, frame = cv2.VideoCapture(video).read()
model = YOLO("yolo11m-obb.pt")

def run_once(imgsz, tiles, overlap, conf=0.35, device="0", obb=True, max_det=200, use_fast_nms=True):
    t0 = time.time()
    merged = tpiv.process_frame_tiled(
        frame, model, imgsz=imgsz, conf=conf, device=device,
        tiles=tiles, overlap_ratio=overlap, nms_iou=0.5,
        obb=obb, classes=[9,10], max_det=max_det, half=True
    )
    t1 = time.time()
    return len(merged), t1 - t0

tests = [
    ("2560, 4x3, 0.25", dict(imgsz=2560, tiles=(4,3), overlap=0.25)),
    ("1280, 4x3, 0.25", dict(imgsz=1280, tiles=(4,3), overlap=0.25)),
    ("1280, 3x2, 0.15", dict(imgsz=1280, tiles=(3,2), overlap=0.15)),
    ("1280, no-tiles",   dict(imgsz=1280, tiles=(1,1), overlap=0.0)),
]

for name, p in tests:
    n, dt = run_once(**p)
    print(f"{name}: {dt:.3f}s, dets={n}")


2560, 4x3, 0.25: 2.937s, dets=816
1280, 4x3, 0.25: 0.254s, dets=817
1280, 3x2, 0.15: 0.129s, dets=715
1280, no-tiles: 0.061s, dets=146


In [5]:
import time, cv2, numpy as np
from ultralytics import YOLO

def timeit(fn, *a, **k):
    t0 = time.time()
    out = fn(*a, **k)
    return out, time.time() - t0

def benchmark_end_to_end(
    frame, model, *,
    do_stab=False, do_detect=True, do_track=True, do_draw=True, do_write=False,
    tiles=(4,3), overlap=0.25, imgsz=1280, conf=0.45, nms_iou=0.5,
    obb=True, classes=[9,10], device="0", max_det=150, min_area=64
):
    times = {}
    out_frame = frame

    # A) Stabilisation
    if do_stab:
        from stabilo import Stabilizer
        stab = Stabilizer()
        stab.set_ref_frame(frame.copy())
        (_, times['stab_set']) = (None, 0.0)   # coût négligeable ici
        _, times['stab'] = timeit(lambda f: (stab.stabilize(f), stab.warp_cur_frame())[1], frame)
        out_frame = stab.warp_cur_frame() or frame

    # B) Détection (tuiles + NMS rapide)
    merged = []
    if do_detect:
        from test_patch_image_video import process_frame_tiled
        def run_detect():
            return process_frame_tiled(
                out_frame, model, imgsz=imgsz, conf=conf, device=device,
                tiles=tiles, num_tiles=None, overlap_ratio=overlap,
                nms_iou=nms_iou, obb=obb, classes=classes,
                max_det=max_det, half=True
            )
        merged, times['detect'] = timeit(run_detect)

        # filtre petite surface pour soulager tracking/drawing
        if min_area and merged:
            keep = []
            for d in merged:
                x1,y1,x2,y2 = d["xyxy"]
                if (x2-x1)*(y2-y1) >= min_area:
                    keep.append(d)
            merged = keep

    # C) Tracking
    tracks_det = None
    if do_track and merged:
        from supervision import Detections
        from supervision.tracker.byte_tracker.core import ByteTrack

        xyxy = np.stack([d["xyxy"] for d in merged], axis=0).astype(np.float32)
        confs = np.array([d["conf"] for d in merged], dtype=np.float32)
        clss  = np.array([d["cls"]  for d in merged], dtype=np.int32)
        dets_sv = Detections(xyxy=xyxy, confidence=confs, class_id=clss)

        H, W = frame.shape[:2]
        fps = 30
        tracker = ByteTrack(
            track_activation_threshold=0.35,
            lost_track_buffer=45,
            minimum_matching_threshold=0.75,
            frame_rate=int(fps)
        )
        tracks_det, times['track'] = timeit(tracker.update_with_detections, dets_sv)

    # D) Drawing
    drawn = out_frame.copy()
    if do_draw and merged:
        from test_patch_image_video import draw_detection, iou_xyxy
        def best_idx_for_track(tr_xyxy, merged_list):
            if not merged_list: return -1
            ious = [iou_xyxy(tr_xyxy, d["xyxy"]) for d in merged_list]
            return int(np.argmax(ious))

        def run_draw():
            nonlocal drawn
            if tracks_det is not None and getattr(tracks_det, "tracker_id", None) is not None:
                for i in range(len(tracks_det)):
                    tid = int(tracks_det.tracker_id[i]) if tracks_det.tracker_id is not None else -1
                    tr_xyxy = tracks_det.xyxy[i].astype(float)
                    best_i = best_idx_for_track(tr_xyxy, merged)
                    if best_i >= 0 and merged[best_i].get("poly") is not None:
                        det_for_draw = {
                            "xyxy": tr_xyxy,
                            "poly": merged[best_i]["poly"],
                            "cls": merged[best_i]["cls"],
                            "conf": merged[best_i]["conf"]
                        }
                        draw_detection(drawn, det_for_draw, None,
                                       show_text=False, show_aabb=False, show_poly=True, id_text=tid)
                    else:
                        x1,y1,x2,y2 = tr_xyxy.astype(int)
                        cv2.rectangle(drawn, (x1,y1), (x2,y2), (0,255,255), 2)
                        cv2.putText(drawn, f"ID {tid}", (x1, max(0,y1-6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 2, cv2.LINE_AA)
            else:
                # juste dessiner les détections
                for d in merged:
                    draw_detection(drawn, d, None, show_text=False, show_aabb=False, show_poly=True)
            return drawn

        drawn, times['draw'] = timeit(run_draw)

    # E) Write (désactive pour mesurer le reste)
    if do_write:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        vw = cv2.VideoWriter("bench_out.mp4", fourcc, 30, (drawn.shape[1], drawn.shape[0]))
        _, times['write'] = timeit(vw.write, drawn)
        vw.release()

    return merged, tracks_det, drawn, times


In [8]:
cap = cv2.VideoCapture("../Dataset/Galatsi_Data_Semester_Project_archive/DJI_0004.MP4")
ret, frame = cap.read(); cap.release()
model = YOLO("yolo11m-obb.pt")

_, _, _, times = benchmark_end_to_end(
    frame, model,
    do_stab=False, do_detect=True, do_track=True, do_draw=True, do_write=False,
    tiles=(4,3), overlap=0.25, imgsz=2560, conf=0.45,
    nms_iou=0.5, obb=True, device="0", max_det=150, min_area=64
)
print(times)


{'detect': 4.417271614074707, 'track': 0.036008358001708984, 'draw': 0.9657042026519775}
